# 🔎 Research Paper Search & Summarization Agent
### Built with LangChain, FAISS, Hugging Face, and Groq (Llama 3.3)

**Project summary**

This project implements an **AI agent** that can search a knowledge base of research
papers and summarize them on demand, using two custom tools:

1. **`search_paper`** — a semantic search tool backed by a **FAISS** vector store, which
   retrieves the most relevant paper(s) for a natural-language query using sentence
   embeddings.
2. **`summarize_paper`** — an abstractive summarization tool built on a Hugging Face
   `distilbart-cnn-12-6` model, wrapped as a LangChain-compatible pipeline.

A **Groq-hosted Llama 3.3** model acts as the reasoning engine (the "brain") of a
LangChain **tool-calling agent**, deciding *when* to search, *when* to summarize, and how
to combine both into a helpful answer for the user — e.g. *"Find me a paper about
transformers and summarize it in simple terms."*

**Architecture**

```
User query
   │
   ▼
LangChain Agent (ChatGroq / Llama 3.3)
   │
   ├──► search_paper(query)      → FAISS similarity search over paper knowledge base
   │
   └──► summarize_paper(text)    → Hugging Face DistilBART summarization pipeline
   │
   ▼
Final natural-language answer
```

**Tech stack:** LangChain, LangChain-Community, LangChain-HuggingFace, LangChain-Groq,
FAISS, Hugging Face Transformers, Sentence-Transformers, PyTorch.

> 💡 This notebook is designed to run top-to-bottom in **Google Colab**. You will need a
> free **Groq API key** (https://console.groq.com/keys) stored as a Colab secret named
> `GROQ_API_KEY` (or as an environment variable if running locally).


## 0. Setup — install dependencies

In [ ]:
# Core LangChain + integration packages, FAISS vector store, and Hugging Face libraries
!pip install -q langchain langchain-community langchain-core langchain-huggingface \
    langchain-groq faiss-cpu sentence-transformers transformers torch --upgrade


In [ ]:
import os
import torch

from langchain_core.tools import tool
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline

from langchain.agents import create_tool_calling_agent, AgentExecutor

from langchain_groq import ChatGroq
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

print("All libraries imported successfully ✅")


## 1. Build the research paper knowledge base

For this demo we use a small, hand-curated set of well-known ML/NLP papers (title +
a short original description of each paper's contribution). In a production version
this could easily be swapped out for a live feed from the **arXiv API** or a folder of
PDFs parsed with a loader such as `PyPDFLoader`.


In [ ]:
# A small demo corpus of well-known research papers.
# (Descriptions below are original short paraphrases, not the papers' actual abstracts.)
papers = [
    {
        "id": "attention-is-all-you-need",
        "title": "Attention Is All You Need (Vaswani et al., 2017)",
        "content": (
            "Introduces the Transformer, a sequence-to-sequence architecture that "
            "relies entirely on self-attention rather than recurrence or convolution. "
            "The attention mechanism lets the model weigh the relevance of every token "
            "against every other token in parallel, which makes training much faster "
            "on modern hardware and became the foundation for almost all later large "
            "language models."
        ),
    },
    {
        "id": "bert",
        "title": "BERT: Pre-training of Deep Bidirectional Transformers (Devlin et al., 2018)",
        "content": (
            "Proposes pre-training a Transformer encoder on masked-language-modeling "
            "and next-sentence-prediction objectives so that it learns bidirectional "
            "context from both left and right of each token. The pre-trained model can "
            "then be fine-tuned with a small additional layer for a wide range of "
            "downstream NLP tasks, achieving state-of-the-art results at the time."
        ),
    },
    {
        "id": "gpt3",
        "title": "Language Models are Few-Shot Learners / GPT-3 (Brown et al., 2020)",
        "content": (
            "Shows that scaling an autoregressive Transformer language model up to "
            "175 billion parameters gives it strong few-shot and even zero-shot "
            "learning abilities: the model can perform new tasks from just a handful "
            "of examples given in its prompt, without any gradient updates or "
            "fine-tuning."
        ),
    },
    {
        "id": "resnet",
        "title": "Deep Residual Learning for Image Recognition / ResNet (He et al., 2015)",
        "content": (
            "Introduces residual (skip) connections that let very deep convolutional "
            "networks be trained without the accuracy degradation normally seen as "
            "depth increases. Residual blocks reformulate layers to learn a small "
            "correction to the identity mapping, which made networks with over 100 "
            "layers practical for image classification."
        ),
    },
    {
        "id": "gan",
        "title": "Generative Adversarial Networks (Goodfellow et al., 2014)",
        "content": (
            "Frames generative modeling as a two-player game between a generator, "
            "which tries to produce realistic fake samples, and a discriminator, "
            "which tries to tell real samples from fake ones. Training both networks "
            "simultaneously drives the generator toward producing increasingly "
            "realistic data, giving rise to the GAN family of generative models."
        ),
    },
    {
        "id": "dropout",
        "title": "Dropout: A Simple Way to Prevent Neural Networks from Overfitting (Srivastava et al., 2014)",
        "content": (
            "Proposes randomly deactivating a fraction of neurons during each training "
            "step so the network cannot rely too heavily on any single unit. This acts "
            "as an efficient approximation to training an ensemble of many smaller "
            "networks, and became a standard regularization technique in deep "
            "learning."
        ),
    },
    {
        "id": "adam",
        "title": "Adam: A Method for Stochastic Optimization (Kingma & Ba, 2014)",
        "content": (
            "Presents the Adam optimizer, which keeps per-parameter running estimates "
            "of the first and second moments of the gradients to adapt the learning "
            "rate for every parameter individually. It combines the benefits of "
            "AdaGrad and RMSProp and remains one of the most widely used optimizers "
            "for training deep neural networks."
        ),
    },
    {
        "id": "word2vec",
        "title": "Efficient Estimation of Word Representations in Vector Space / word2vec (Mikolov et al., 2013)",
        "content": (
            "Introduces two simple and efficient neural architectures, "
            "continuous-bag-of-words and skip-gram, for learning dense vector "
            "representations of words from large amounts of unlabeled text. The "
            "resulting word vectors capture semantic and syntactic relationships, "
            "famously supporting analogy arithmetic such as king minus man plus "
            "woman equals queen."
        ),
    },
    {
        "id": "rag",
        "title": "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks / RAG (Lewis et al., 2020)",
        "content": (
            "Combines a pretrained sequence-to-sequence generator with a dense "
            "retriever over a large external document index, so the model can pull in "
            "relevant passages at generation time instead of relying only on facts "
            "memorized in its parameters. This reduces hallucination and lets the "
            "model's knowledge be updated by changing the retrieval index rather than "
            "retraining it."
        ),
    },
    {
        "id": "clip",
        "title": "Learning Transferable Visual Models From Natural Language Supervision / CLIP (Radford et al., 2021)",
        "content": (
            "Trains an image encoder and a text encoder jointly on hundreds of "
            "millions of image-caption pairs using a contrastive objective, so that "
            "matching image-text pairs end up close together in a shared embedding "
            "space. The resulting model performs strong zero-shot image "
            "classification simply by comparing an image embedding to text prompts "
            "describing candidate classes."
        ),
    },
]

documents = [
    Document(page_content=f"{p['title']}. {p['content']}", metadata={"id": p["id"], "title": p["title"]})
    for p in papers
]

print(f"Knowledge base built with {len(documents)} papers ✅")


## 2. Embed the papers and build a FAISS vector store

We embed each paper with a lightweight `sentence-transformers` model and index the
vectors with **FAISS** so we can do fast semantic (meaning-based, not just
keyword-based) similarity search.


In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(documents, embeddings)

# Save the index to disk so it can be reloaded later without re-embedding
vectorstore.save_local("faiss_paper_index")

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("FAISS vector store built and saved to 'faiss_paper_index' ✅")


## 3. Tool #1 — `search_paper`

A LangChain `@tool` that wraps the FAISS retriever. Given a natural-language query, it
returns the most relevant paper(s) (title + description) from the knowledge base.


In [ ]:
@tool
def search_paper(query: str) -> str:
    """Search the research paper knowledge base (FAISS vector store) and return the
    most relevant paper(s), including title and a short description, for the given
    query."""
    results = retriever.invoke(query)
    if not results:
        return "No relevant papers were found in the knowledge base."

    formatted = []
    for i, doc in enumerate(results, start=1):
        title = doc.metadata.get("title", "Unknown title")
        formatted.append(f"{i}. {title}\n{doc.page_content}")
    return "\n\n".join(formatted)


# Quick manual test (does not require any API key, runs fully locally)
print(search_paper.invoke({"query": "How does self-attention work in transformers?"}))


## 4. Summarization pipeline

We use `sshleifer/distilbart-cnn-12-6`, a distilled BART model fine-tuned for
abstractive summarization, wrapped in a small custom class so it plugs cleanly into
LangChain (as both a callable pipeline and, optionally, as a `HuggingFacePipeline`
LLM).


In [ ]:
# Initialize the summarization model and tokenizer
model_name = "sshleifer/distilbart-cnn-12-6"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


class CustomSummarizationPipeline:
    """A minimal callable wrapper around a Hugging Face seq2seq model that mimics
    the output format of `transformers.pipeline('summarization')`, so it can be reused
    both as a plain Python function and inside a LangChain `HuggingFacePipeline`."""

    def __init__(self, model, tokenizer, device: str | None = None):
        self.model = model
        self.tokenizer = tokenizer
        self.task = "summarization"
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

    def __call__(self, text: str, **kwargs):
        inputs = self.tokenizer(text, return_tensors="pt", max_length=1024, truncation=True).to(self.device)
        generation_kwargs = {
            "max_new_tokens": 150,
            "min_length": 30,
            "num_beams": 4,
            "do_sample": False,
            "early_stopping": True,
            **kwargs,  # allow callers to override any default
        }
        summary_ids = self.model.generate(inputs["input_ids"], **generation_kwargs)
        summary_text = self.tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        return [{"summary_text": summary_text}]


# Callable summarizer instance
summarizer = CustomSummarizationPipeline(model, tokenizer)

# Also expose it as a LangChain LLM in case we want to use it directly in a chain
summarization_llm = HuggingFacePipeline(pipeline=summarizer)

print("Summarization pipeline ready ✅")


## 5. Tool #2 — `summarize_paper`

In [ ]:
@tool
def summarize_paper(text: str) -> str:
    """Summarize a piece of text (for example, a paper description returned by
    search_paper) into a short, easy-to-read 2-3 sentence summary."""
    result = summarizer(text)
    return result[0]["summary_text"]


# Quick manual test
sample_text = documents[0].page_content
print("Original:\n", sample_text)
print("\nSummary:\n", summarize_paper.invoke({"text": sample_text}))


## 6. Connect the agent's reasoning engine — Groq (Llama 3.3)

The agent needs an LLM capable of *tool calling* (deciding which tool to use and with
what arguments). We use **Groq**, which serves Llama 3.3 with very low latency, via
`langchain-groq`.

Set your key as a Colab secret called `GROQ_API_KEY` (🔑 icon in the left sidebar), or
as a plain environment variable if you're running this notebook locally.


In [ ]:
# Try to load the key from Colab secrets first, then fall back to a local
# environment variable / interactive prompt so this notebook also works outside Colab.
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except Exception:
    if "GROQ_API_KEY" not in os.environ:
        import getpass
        os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

print("Groq LLM initialized ✅")


## 7. Build the tool-calling agent

We give the LLM access to both tools (`search_paper`, `summarize_paper`) and let it
decide, based on the user's request, whether it needs to search, summarize, both, or
neither.


In [ ]:
tools = [search_paper, summarize_paper]

system_prompt = (
    "You are a helpful research assistant. You have access to two tools:\n"
    "1. search_paper - looks up relevant papers in a knowledge base.\n"
    "2. summarize_paper - condenses a piece of text into a short summary.\n\n"
    "When the user asks about a topic, use search_paper to find relevant papers first. "
    "If the user also asks for a summary, pass the retrieved paper description into "
    "summarize_paper and present the final summary in clear, simple language. "
    "Always mention the title(s) of the paper(s) you used."
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("Agent ready ✅")


## 8. Try it out

In [ ]:
response = agent_executor.invoke({
    "input": "Find me a paper about transformer architectures and summarize it in simple terms."
})
print("\nFINAL ANSWER:\n", response["output"])


In [ ]:
response = agent_executor.invoke({
    "input": "What paper introduced word embeddings, and what did it show?"
})
print("\nFINAL ANSWER:\n", response["output"])


In [ ]:
response = agent_executor.invoke({
    "input": "I'm interested in how neural networks avoid overfitting. Search for a relevant paper and give me a one-line takeaway."
})
print("\nFINAL ANSWER:\n", response["output"])


## 9. Conclusion & possible extensions

This notebook demonstrates a complete, working **retrieval + summarization agent**
built with LangChain:

- A **FAISS**-backed semantic search tool over a small research-paper knowledge base.
- A **Hugging Face** abstractive summarization tool.
- A **Groq/Llama-3.3** tool-calling agent that autonomously decides how to combine
  both tools to answer a user's question.

**Possible extensions for a v2:**
- Replace the hand-curated corpus with a live **arXiv API** ingestion pipeline so the
  knowledge base can grow automatically.
- Add a `PyPDFLoader`-based tool so users can upload their own PDFs to search over.
- Add conversational memory so the agent can handle multi-turn follow-up questions
  ("summarize the *second* one instead").
- Wrap the agent in a simple **Streamlit** or **Gradio** app for a shareable demo UI.
- Cache embeddings and reload the saved `faiss_paper_index` instead of recomputing it
  on every run.

---
*Project built as part of an internship assignment — feel free to fork and extend!*
